# Linguistic Diversity — a tour

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabriceyhc/linguistic-diversity/blob/main/examples/demo.ipynb)

Most diversity work in NLP counts **surface forms** — type-token ratio, distinct-n,
self-BLEU. This library measures whether the **meanings** differ, using
similarity-sensitive Hill numbers borrowed from ecology.

The difference is not academic: the two families of metric can rank the same pair of
corpora in **opposite** directions. This notebook demonstrates that, then walks through
what each metric measures and when to reach for it.

Runtime: about 5 minutes on a free Colab CPU instance (a few hundred MB of models are
downloaded on first use). A GPU is optional and is used automatically if present.

## Setup

In [ ]:
# Colab and other fresh environments need the package plus a spaCy pipeline and
# a few NLTK corpora. Locally this is a no-op if you already have them.
import importlib.util
import subprocess
import sys


def sh(*args: str) -> None:
    print("$", " ".join(args))
    subprocess.run(args, check=True)


if importlib.util.find_spec("linguistic_diversity") is None:
    # [phonological] pulls in pyphen, pronouncing and g2p_en, which the
    # Rhythmic and Phonemic metrics need. Without it those two raise ImportError.
    sh(sys.executable, "-m", "pip", "install", "-q", "linguistic-diversity[phonological]")

if importlib.util.find_spec("en_core_web_sm") is None:
    sh(sys.executable, "-m", "spacy", "download", "en_core_web_sm")

sh(
    sys.executable, "-m", "nltk.downloader", "-q",
    "stopwords", "cmudict", "averaged_perceptron_tagger_eng", "punkt",
)
print("\nready")

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import linguistic_diversity as ld

print("linguistic-diversity", ld.__version__)

import torch

print("device:", "cuda" if torch.cuda.is_available() else "cpu")

## 1. Why not just measure lexical diversity?

Two sets, deliberately matched: **5 documents, 30 words, 30 token species each**. Only
the content differs.

- **Set A** — every word is unique. One idea, restated five times.
- **Set B** — the word *run* five times, in five unrelated senses.

In [ ]:
lexically_diverse = [
    "a violent tempest wrecked our village",
    "the fierce gale devastated their settlement",
    "that savage hurricane destroyed this community",
    "an intense cyclone flattened every township",
    "some brutal windstorm ruined nearby neighborhoods",
]

semantically_diverse = [
    "she went for a morning run",           # jogging
    "he will run the entire company",       # to manage
    "a run appeared in her stocking",       # a tear
    "the program failed to run correctly",  # to execute
    "they scored the winning run today",    # a baseball point
]

for name, corpus in [("Set A", lexically_diverse), ("Set B", semantically_diverse)]:
    words = [w for s in corpus for w in s.split()]
    print(f"{name}: {len(corpus)} documents, {len(words)} words, {len(set(words))} unique")

### What the lexical metrics say

All three ship with the library, so you can reproduce this rather than take it on faith.
Note self-BLEU is **inverted**: it measures overlap, so *lower* means more diverse.

In [ ]:
import pandas as pd

from linguistic_diversity import DistinctN, SelfBLEU, TypeTokenRatio

lexical = {
    "type-token ratio": TypeTokenRatio(),
    "distinct-1": DistinctN({"n": 1}),
    "distinct-2": DistinctN({"n": 2}),
    "self-BLEU (lower = diverse)": SelfBLEU(),
}

rows = []
for name, metric in lexical.items():
    a, b = metric(lexically_diverse), metric(semantically_diverse)
    better_is_lower = name.startswith("self-BLEU")
    verdict = ("Set A" if (a < b) == better_is_lower else "Set B") if a != b else "tie"
    rows.append({"measure": name, "Set A": round(a, 3), "Set B": round(b, 3),
                 "says more diverse": verdict})

pd.DataFrame(rows)

Set A scores **perfectly on every lexical measure** — type-token ratio 1.000, and a
self-BLEU of 0.000 meaning literally zero n-gram overlap between its sentences.

Now ask what the sentences actually *mean*.

In [ ]:
from linguistic_diversity import DocumentSemantics

doc_semantics = DocumentSemantics()

a = doc_semantics(lexically_diverse)
b = doc_semantics(semantically_diverse)

print(f"Set A: {a:.2f} effective meanings out of 5")
print(f"Set B: {b:.2f} effective meanings out of 5")
print(f"\nSet B is {b / a:.1f}x more semantically diverse — the opposite ranking.")

Set A conveys **one proposition five times**. Set B looks repetitive to a lexical metric
because *run* recurs, but each occurrence is a different word sense.

If you are selecting training data, deduplicating a corpus, or scoring generation
diversity, the lexical metrics will happily accept Set A as maximally diverse. It isn't.

## 2. The other dimensions

Diversity is not one number. The library measures four linguistic branches, and they are
genuinely independent signals — a corpus can be semantically varied yet syntactically
monotonous.

In [ ]:
from linguistic_diversity import (
    DependencyParse,
    PartOfSpeechSequence,
    Phonemic,
    Rhythmic,
    TokenSemantics,
)

# Rhythmic and Phonemic need the [phonological] extras. If the install cell
# above was skipped they raise ImportError, so report that rather than stopping.
candidates = [
    ("Semantic", "TokenSemantics", TokenSemantics),
    ("Semantic", "DocumentSemantics", lambda: doc_semantics),
    ("Syntactic", "DependencyParse", DependencyParse),
    ("Morphological", "PartOfSpeechSequence", PartOfSpeechSequence),
    ("Phonological", "Rhythmic", Rhythmic),
    ("Phonological", "Phonemic", Phonemic),
]

rows = []
for dimension, name, build in candidates:
    try:
        metric = build()
    except ImportError as exc:
        print(f"skipping {name}: {exc.args[0].splitlines()[0]}")
        continue
    rows.append({
        "dimension": dimension,
        "metric": name,
        "Set A": round(metric(lexically_diverse), 2),
        "Set B": round(metric(semantically_diverse), 2),
    })

results = pd.DataFrame(rows)
results["gap"] = (results["Set B"] - results["Set A"]).round(2)
results

Two things worth noticing:

- **Syntax separates these sets most sharply.** Set A repeats one grammatical frame
  (`DET ADJ NOUN VERB DET NOUN`) five times, while Set B mixes a clause, an imperative
  and a noun phrase. That is information no semantic metric gives you.
- **Phonology tracks neither.** Sound patterns are largely independent of meaning, which
  is exactly why the dimensions are measured separately rather than mashed into one score.

Let's see the syntax difference directly.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

for name, corpus in [("Set A", lexically_diverse), ("Set B", semantically_diverse)]:
    print(f"{name}")
    for sentence in corpus[:3]:
        tags = [t.pos_ for t in nlp(sentence)]
        print(f"    {sentence:45s} {' '.join(tags)}")
    print()

## 3. One number: the universal metric

`UniversalLinguisticDiversity` combines the branches hierarchically — geometric mean
within a branch, weighted combination across branches — and can show its working.

In [ ]:
from linguistic_diversity import UniversalLinguisticDiversity

universal = UniversalLinguisticDiversity()

breakdown = {}
for name, corpus in [("Set A", lexically_diverse), ("Set B", semantically_diverse)]:
    detailed = universal.get_detailed_scores(corpus)
    breakdown[name] = {**detailed["branches"], "UNIVERSAL": detailed["universal"]}

pd.DataFrame(breakdown).round(2)

Presets adjust the branch weights for different questions — `semantic_focus` for content
analysis, `structural_focus` for grammatical work, `conservative` when a single weak
dimension should drag the score down.

In [ ]:
from linguistic_diversity import get_preset_config

for preset in ["balanced", "semantic_focus", "structural_focus", "conservative"]:
    config = get_preset_config(preset)
    weights = {k.replace("_weight", ""): v for k, v in config.items() if k.endswith("_weight")}
    branches = {k: v for k, v in weights.items()
                if k in {"semantic", "syntactic", "morphological", "phonological"}}
    print(f"{preset:18s} strategy={config.get('strategy', 'hierarchical'):14s} {branches}")

## 4. Large corpora

Exact diversity needs an O(n²) similarity matrix, which stops being practical quickly.
`estimate_diversity()` samples at increasing sizes, fits a growth curve, and extrapolates
— the same idea as a rarefaction curve in ecology or Heaps' law in linguistics.

In [ ]:
import random

random.seed(0)

# A synthetic corpus with a known structure: 40 distinct topics, restated many times.
topics = [
    "the storm damaged the coastal town",
    "researchers published a new finding",
    "the team won the championship match",
    "prices rose sharply this quarter",
    "the software update fixed several bugs",
]
templates = ["{}", "Reportedly, {}.", "It is said that {}.", "Sources confirm {}."]

large_corpus = [t.format(topic) for topic in topics for t in templates for _ in range(40)]
random.shuffle(large_corpus)

print(f"corpus size: {len(large_corpus)} documents")

In [ ]:
result = doc_semantics.estimate_diversity(
    large_corpus, base_sample_size=25, max_sample_size=100, num_trials=2, verbose=False
)

print(f"method      : {result.method}")
print(f"estimate    : {result.diversity:.2f} +/- {result.std:.2f}")
print(f"growth model: {result.model}  (fit RMSE {result.fit_rmse:.4f})")
print(f"measured at : {result.sample_sizes}")
print(f"observed    : {[round(m, 2) for m in result.sample_means]}")

The estimate never builds the full n×n matrix — the largest sample it measured is shown
above. `result.plot()` visualises the sampling points and the fitted curve if matplotlib
is available.

## 5. Choosing an embedder

`DocumentSemantics` defaults to `all-mpnet-base-v2`, but the encoder matters. Which one
is best is an empirical question, and the repository answers it in
[`benchmarks/embedder_selection/`](https://github.com/fabriceyhc/linguistic-diversity/tree/main/benchmarks/embedder_selection):
ten open-weight models scored against corpora with known ground-truth diversity and 600
human-judged response sets.

The headline finding is worth knowing before you pick one: **MTEB PairClassification
predicts performance here well (r = +0.879), better than STS.** Any model can be
swapped in via config.

In [ ]:
# Swapping the encoder is a one-line change. (This downloads another model.)
alternative = DocumentSemantics({"model_name": "BAAI/bge-base-en-v1.5"})

print(f"{'model':28s} {'Set A':>7s} {'Set B':>7s} {'ratio':>7s}")
for label, metric in [("all-mpnet-base-v2", doc_semantics), ("bge-base-en-v1.5", alternative)]:
    a, b = metric(lexically_diverse), metric(semantically_diverse)
    print(f"{label:28s} {a:7.2f} {b:7.2f} {b / a:7.2f}")

## 6. Try your own text

Replace the two lists below. Keep them the same length and roughly the same total word
count if you want the numbers to be directly comparable — diversity is bounded above by
the number of species, so unequal sizes have unequal ceilings.

In [ ]:
your_corpus_a = [
    "the meeting ran long again today",
    "our discussion extended past the hour",
    "the session continued well beyond schedule",
]

your_corpus_b = [
    "the meeting ran long again today",
    "she prefers black coffee in the morning",
    "the bridge closes for repairs on tuesday",
]

comparison = []
for label, corpus in [("A", your_corpus_a), ("B", your_corpus_b)]:
    comparison.append({
        "corpus": label,
        "type-token ratio": round(TypeTokenRatio()(corpus), 3),
        "document semantics": round(doc_semantics(corpus), 2),
        "dependency parse": round(DependencyParse()(corpus), 2),
        "universal": round(universal(corpus), 2),
    })

pd.DataFrame(comparison)

## Where to go next

- **[README](https://github.com/fabriceyhc/linguistic-diversity)** — full metric table and
  the reasoning behind each default
- **[Universal metric guide](https://github.com/fabriceyhc/linguistic-diversity/blob/main/docs/universal-metric.md)** —
  aggregation strategies and weighting in depth
- **[Troubleshooting](https://github.com/fabriceyhc/linguistic-diversity/blob/main/docs/troubleshooting.md)** —
  including how to spot a metric that has saturated rather than measured
- **[Experiments](https://github.com/fabriceyhc/linguistic-diversity-experiments)** —
  authorship verification, dementia detection, diversity-based data selection

```bibtex
@software{linguistic_diversity_2026,
  title={Linguistic Diversity: Modernized Implementation of Similarity-Sensitive Hill Numbers for NLP},
  author={Harel-Canada, Fabrice},
  year={2026},
  url={https://github.com/fabriceyhc/linguistic-diversity}
}
```